# Llama Ablation — No NEW_SENSE

**Ablation purpose:** Test forced-choice sense assignment — the model must always choose the
closest ID from the provided list and is explicitly forbidden from returning `NEW_SENSE`.
The JSON schema keeps `sense_id` **and** `explanation` (same as baseline) so this ablation
tests *only* the open-set vs closed-set factor.

All other pipeline components are identical to `Llama_sense.ipynb`:
same model (`llama4`), temperature, chunk selection, processing loop, and writers.

Outputs are written with the `Llama4_nonew` origin label so they never overwrite baseline files.

## Model Information

| Property | Value |
|----------|-------|
| **Model** | Meta Llama 4 Scout |
| **Ollama Tag** | `llama4:latest` |
| **Temperature** | 0.0 |
| **Integration** | langchain-ollama |
| **Prompt Variant** | `Llama4_nonew` — sense_id + explanation, NEW_SENSE forbidden |

In [ ]:
from config import ANNOTATION_CHUNKS, DATA_DIR, OUTPUT_DIR
from data_loader import load_sense_repo_by_round
from process_senses import process_senses_with_chain, default_build_senses_block, parse_model_output
from writers import CustomWebAnnoTSVWriter, InceptionWebAnnoTSVWriter
import time

# Annotation round to use (1 = old/first round, 2 = current/second round)
ROUND = 2

# Set model origin for traceability (ablation: no NEW_SENSE)
ORIGIN_LLM = "Llama4_nonew"


In [ ]:
# Load sense repository and select corpus chunk
senses_df = load_sense_repo_by_round(round_number=ROUND)
print(f"Loaded sense repo for round {ROUND}: {len(senses_df)} senses")

from webanno_spacy_converter.parsers.tsv_parser_v3 import WebAnnoLEXISParser

chunks = [(b, e, (DATA_DIR / fname)) for (b, e, fname) in ANNOTATION_CHUNKS]
print("Available chunks (index, begin, end, file):",
      [(i, b, e, p.name) for i, (b, e, p) in enumerate(chunks)])

chunk_idx = 0  # 0 = 1-500, 1 = 501-1000, 2 = 1001-1500, ...
chunk_begin, chunk_end, selected_path = chunks[chunk_idx]
print(f"Using chunk #{chunk_idx}: {selected_path.name} -> ({chunk_begin}, {chunk_end})")

parser = WebAnnoLEXISParser(selected_path)
sentences = parser.parse()

begin, end = chunk_begin, chunk_end

# For ablation runs, use a small slice by default so the notebook finishes quickly.
# Set test = False to run the full chunk.
test = True
if test:
    tb, te = 0, 20
    sentences = sentences[tb:te]
    begin = chunk_begin + tb
    end = chunk_begin + te - 1
    print(f"Using subset offsets {tb}:{te} -> absolute sentence range {begin}-{end}")
else:
    print(f"Using full chunk sentence range {begin}-{end}")


In [ ]:
from langchain_ollama import ChatOllama
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Ablation: NEW_SENSE is explicitly forbidden.
# The model must always choose the closest ID from the provided list.
# JSON schema keeps sense_id + explanation (same as baseline) to isolate only the open-set factor.
system_message = """
Vi ste ekspert za leksiku i semantiku. Na osnovu konteksta rečenice i liste validnih značenja ciljne reči,
vaš zadatak je da identifikujete ono značenje koje se najpreciznije koristi u datom kontekstu.

Odgovor mora biti u strogo definisanom JSON formatu:

{{
  "sense_id": "<TAČAN ID iz liste>",
  "explanation": "<kratko obrazloženje u jednoj ili dve rečenice>"
}}

VAŽNO (ABLACIJA: bez NEW_SENSE):
- "NEW_SENSE" NIJE DOZVOLJEN u ovoj verziji
- sense_id MORA biti IDENTIČAN jednom od ponuđenih ID-jeva iz liste (npr. "ENG30-00551215-n")
- Ako nijedno značenje nije savršeno, izaberite NAJBLIŽE značenje iz liste
- NIKADA ne koristite brojeve poput "1", "2", "značenje 1" itd.
- Ne dodajete nikakav tekst van JSON strukture
"""

user_prompt = """
Kontekst rečenice (ciljna reč je označena HTML tagom <b>...</b>):
"{sentence}"

Ciljna reč: "{word}"

Lista mogućih značenja:
{senses_block}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_message),
    ("user", user_prompt)
])

llm = ChatOllama(
    model="llama4",
    temperature=0.0,
)
parser = StrOutputParser()
chain = prompt | llm | parser

# Confirm origin label matches single source of truth.
assert ORIGIN_LLM == "Llama4_nonew"


In [ ]:
# --- DEBUG: Preview raw model outputs for early items ---
# Set DEBUG_PREVIEW = False to inspect a few raw model responses before the full run.
DEBUG_PREVIEW = False

if DEBUG_PREVIEW:
    from notebook_utils import debug_preview_chain
    debug_preview_chain(sentences, senses_df, chain, default_build_senses_block)


In [ ]:
# Annotate senses using the shared utility
start_time = time.time()
sentences = process_senses_with_chain(
    sentences,
    senses_df,
    chain,
    ORIGIN_LLM,
    build_senses_block=default_build_senses_block,
    parse_json_response_clean=parse_model_output
)
end_time = time.time()
print(f"Processed sentences {begin} to {end} in {end_time - start_time:.2f} seconds.")


In [ ]:
# Save outputs for the selected round and chunk
ROUND_SUFFIX = f"_round{ROUND}"
writer = CustomWebAnnoTSVWriter(sentences)
writer.save(OUTPUT_DIR / f"LexiSense_{begin:04d}_{end:04d}_{ORIGIN_LLM}{ROUND_SUFFIX}.tsv")


In [ ]:
# Write Inception-compatible output (for annotation import)
incept_writer = InceptionWebAnnoTSVWriter(sentences)
incept_writer.save(OUTPUT_DIR / f"LexiSense_Inception_{begin:04d}_{end:04d}_{ORIGIN_LLM}{ROUND_SUFFIX}.tsv")


## Log processing time and completion

In [ ]:
with open(OUTPUT_DIR / f"llama_{ORIGIN_LLM}_round{ROUND}.log", "a", encoding="utf-8") as f:
    f.write(f"Chunk {chunk_idx} | processed sentences {begin} to {end}\n")
    f.write(f"{ORIGIN_LLM} took {end_time - start_time:.2f} seconds\n")
    f.write(f"Or minutes: {(end_time - start_time)/60:.2f}\n")
print("Done.")
